# 12.01 Langfuse — 오픈소스 LLM 관측 플랫폼

[Langfuse](https://langfuse.com) 는 **self-host 가능**한 오픈소스 LLM 관측 플랫폼이다.
`06_langsmith/` 가 다루는 LangSmith 와 개념적으로 거의 1:1 대응하지만, 데이터를 **자기 인프라에 두고 싶은 조직**(컴플라이언스·온프레미스·에어갭)에서 주로 선택된다.

## 학습 목표

- `langfuse` 4.x 의 `CallbackHandler` 를 `create_agent(...)` / `invoke(..., config={"callbacks":[...]})` 에 주입해 자동 트레이싱
- Langfuse 의 3대 개념(**trace / score / dataset**) 을 LangSmith 의 **run / feedback / dataset** 과 매핑
- **self-host (Docker Compose) vs Cloud** 티어 선택 기준
- 모델별 **cost tracking** 자동 계산 (프로젝트 설정의 `Model pricing`)
- LangSmith 와 **동시 운용**: callback 두 개를 한 번에 주입

## LangSmith 와의 관계

| 측면 | LangSmith | Langfuse |
|------|-----------|----------|
| 배포 | SaaS (cloud) | self-host + cloud 양쪽 |
| 라이선스 | 상용 | MIT + Commercial (EE) |
| LangChain 자동 계측 | env var 3줄 | `CallbackHandler` 주입 |
| 데이터 소재 | LangChain Inc. | 사용자 선택 |
| OTel 프로토콜 | exporter 지원 (최근) | 4.x 내부가 OTel 기반 |

둘은 **동시에 붙일 수 있다** (§12.01.6). 마이그레이션 기간이나 조직별 분할 운영에 유용하다.

## 12.01.1 환경 설정

필요 패키지: `langfuse` (4.x 부터 LangChain 통합이 **본체 안에 포함** — 별도 `langfuse-langchain` 패키지 없음).
`.env` 에 세 값이 있어야 한다.

```dotenv
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=https://cloud.langfuse.com   # self-host 면 http://localhost:3000 등
```

Cloud 계정은 https://cloud.langfuse.com → Project → Settings → API Keys 에서 발급.

In [ ]:
# !pip install -U langfuse langchain langchain-openai

import os
from dotenv import load_dotenv
load_dotenv(override=True)

# 이 노트북이 요구하는 키/서비스만 probe
assert os.environ.get("LANGFUSE_PUBLIC_KEY"), "LANGFUSE_PUBLIC_KEY 미설정"
assert os.environ.get("LANGFUSE_SECRET_KEY"), "LANGFUSE_SECRET_KEY 미설정"
assert os.environ.get("LANGFUSE_HOST"),        "LANGFUSE_HOST 미설정 (cloud 는 https://cloud.langfuse.com)"
assert os.environ.get("OPENAI_API_KEY"),       "OPENAI_API_KEY 미설정"

print("Langfuse host :", os.environ["LANGFUSE_HOST"])

## 12.01.2 기본 사용 — `CallbackHandler` 주입

Langfuse 4.x 는 내부가 **OpenTelemetry 기반**으로 재작성됐다. `CallbackHandler` 를 LangChain callbacks 로 넘기면, LangChain 이 발생시키는 chain/tool/llm 이벤트가 Langfuse span 으로 자동 변환된다.

- **Agent 레벨 주입**: `create_agent(..., config={"callbacks":[handler]})` 는 v1 에서 권장되지 않는다 (agent 자체는 config 를 받지 않음).
- **호출 레벨 주입**: `agent.invoke(..., config={"callbacks":[handler]})` 가 표준 경로.
- 환경변수 세 개가 설정돼 있으면 `CallbackHandler()` 는 **인자 없이** 생성 가능.

In [ ]:
from langfuse.langchain import CallbackHandler
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 반환한다."""
    return f"{city}: 맑음, 21°C"

handler = CallbackHandler()   # env var 자동 픽업

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[get_weather],
    system_prompt="당신은 친절한 날씨 봇입니다.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "서울 날씨 알려줘"}]},
    config={"callbacks": [handler]},
)
result["messages"][-1].content

실행 후 Langfuse UI (`$LANGFUSE_HOST` 브라우저 접속) → 좌측 `Tracing` → 방금 만든 trace 가 한 행으로 나타난다.
- **Trace Tree** 에 `ChatOpenAI → tool: get_weather → ChatOpenAI` 구조가 waterfall 로 그려진다.
- 각 span 은 input/output/latency/tokens/cost 를 자동 캡처한다.

### trace 메타 정보 부착

LangSmith 의 `run_name` / `tags` / `metadata` 에 해당하는 항목은 Langfuse 에서도 동일하게 지원된다.
`invoke(config=...)` 에 `run_name` / `tags` / `metadata` 를 넣으면 Langfuse handler 가 그대로 span 속성으로 밀어 넣는다.

In [ ]:
config = {
    "callbacks": [handler],
    "run_name": "weather-bot:busan-request",
    "tags": ["env:dev", "feature:weather"],
    "metadata": {
        "user_id": "u_00123",
        "session_id": "s_abc",
        "app_version": "0.5.0",
    },
}
agent.invoke(
    {"messages": [{"role": "user", "content": "부산 날씨 알려줘"}]},
    config=config,
)

UI 에서 `Filter → Tags contains env:dev` 또는 `Metadata.user_id = u_00123` 으로 검색 가능하다.

## 12.01.3 trace · score · dataset — 3대 개념

Langfuse 의 핵심 엔티티 세 개를 LangSmith 와 1:1 로 매핑하면 이렇다.

| Langfuse | LangSmith | 의미 |
|----------|-----------|------|
| **trace** | run | 한 번의 실행 = 여러 span 의 루트 |
| **score** | feedback | 정답률·thumbs-up·latency 등 trace 에 붙는 평가값 |
| **dataset** | dataset | 입력·기대출력 쌍의 집합. 회귀 테스트 / 평가 실행의 소스 |

점수를 코드로 붙이는 전형적 패턴은 다음과 같다. `get_client()` 는 싱글턴 Langfuse 클라이언트를 반환하며, `create_score()` / `create_dataset()` 같은 REST API wrapper 를 제공한다.

In [ ]:
from langfuse import get_client

client = get_client()

# 1) dataset 생성 (평가 셋의 컨테이너)
ds_name = "weather-bot-smoke"
client.create_dataset(
    name=ds_name,
    description="weather-bot 회귀 입력 샘플",
    metadata={"owner": "qa-team"},
)

# 2) dataset item 추가
client.create_dataset_item(
    dataset_name=ds_name,
    input={"messages": [{"role": "user", "content": "서울 날씨"}]},
    expected_output="서울: 맑음, 21°C",
)

print("dataset 준비 완료:", ds_name)

### score — trace 단위로 평가값 쓰기

방금 실행한 agent 호출에 `helpfulness` 같은 커스텀 지표를 남긴다.
`handler.last_trace_id` 같은 헬퍼는 없고, Langfuse 4.x 는 OTel span context 로 trace_id 를 조회한다. 가장 단순한 경로는 `@observe` 데코레이터로 래핑해 trace_id 를 직접 받는 것이다.

In [ ]:
from langfuse import observe

@observe(name="weather-pipeline")
def run_weather(city: str) -> str:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": f"{city} 날씨"}]},
        config={"callbacks": [handler]},
    )
    # 현재 trace 에 점수 부착
    client.score_current_trace(
        name="helpfulness",
        value=0.9,
        comment="수동 라벨링",
    )
    return result["messages"][-1].content

run_weather("제주")

UI 의 trace 상세 화면 오른쪽 `Scores` 탭에 `helpfulness: 0.9` 가 꽂힌다.
프로젝트 레벨에서는 `Evaluations → Scores` 에서 시계열 집계가 가능하다.

## 12.01.4 self-host vs Cloud

Langfuse 의 차별점은 **self-host** 경로다. 선택 기준을 정리하면 다음과 같다.

| 항목 | Cloud | Self-host (Docker Compose) |
|------|-------|----------------------------|
| 초기 세팅 | API Key 발급 1분 | Postgres + ClickHouse + Redis + Web + Worker (Compose 로 번들) |
| 데이터 | EU/US 리전 | 사용자 인프라 (온프레미스·VPC) |
| 가격 | Trace 수 기반 티어 | 인프라 비용만 (EE 기능은 Pro 라이선스) |
| 업데이트 | 자동 | `git pull && docker compose up -d` |
| 컴플라이언스 | SOC2 / GDPR | 사용자 책임 |

### 로컬 self-host 요약 (참조용)

```bash
git clone https://github.com/langfuse/langfuse.git
cd langfuse
cp .env.dev.example .env
docker compose up -d
# http://localhost:3000 접속 → 초기 사용자 생성 → 프로젝트 생성 → API Keys 발급
```

이후 노트북의 `.env` 를 다음처럼 교체하면 끝.

```dotenv
LANGFUSE_HOST=http://localhost:3000
LANGFUSE_PUBLIC_KEY=pk-lf-local-...
LANGFUSE_SECRET_KEY=sk-lf-local-...
```

> self-host 컴포넌트 중 **ClickHouse 가 trace 저장소**다. 스토리지/백업 정책은 여기 기준으로 설계.

## 12.01.5 Cost tracking — 모델별 단가 자동 계산

Langfuse 는 내장 `Model pricing` 테이블을 가지며, span 의 `model` 필드와 매칭해 `total_cost` 를 자동 계산한다.
- 내장 테이블: OpenAI / Anthropic / Google / Mistral / Cohere / Groq / Bedrock 등 주요 모델의 $/1K tokens.
- 커스텀 모델(사내 fine-tuned 등)은 **UI 의 `Settings → Model pricing`** 에서 정규식 매치 규칙 + input/output 단가를 추가.
- REST API 로도 가능: `client.api.models.create(...)` (SDK 4.x, `/api/public/models`).

### UI 에서 보는 법

- Trace 상세 → 각 LLM span 의 `Usage` 섹션에 `input_tokens · output_tokens · cost` 표시.
- Dashboard → `Costs` 차트로 시계열 / 사용자별 / 태그별 비용 분할.

> 단가가 누락된 모델은 `cost = null` 로 기록된다. cost 가 `null` 이면 dashboard 집계에서 빠지므로, 프로덕션 전 모델 ID 단가가 매칭되는지 한 번 확인할 것.

## 12.01.6 LangSmith 와 동시 운용

LangChain callbacks 는 **리스트**다. 두 개를 한 번에 넣으면 두 백엔드에 **같은 trace 가 이중으로 전송**된다.
마이그레이션 기간 / 조직별 분리 운영 / A/B 비교 등에 쓴다.

이 셀은 `LANGSMITH_API_KEY` 와 `LANGSMITH_TRACING=true` 가 모두 있을 때만 LangSmith 쪽도 실제로 기록한다. 환경변수가 없으면 LangSmith 부분은 no-op.

In [ ]:
# LangSmith 는 callback 객체 없이 env var 만으로 동작하는 것이 기본이지만,
# 명시적 핸들러를 쓰고 싶으면 아래처럼 LangChainTracer 를 직접 주입한다.
from langchain_core.tracers.langchain import LangChainTracer

callbacks = [handler]   # Langfuse 는 항상
if os.environ.get("LANGSMITH_API_KEY") and os.environ.get("LANGSMITH_TRACING") == "true":
    callbacks.append(LangChainTracer(project_name=os.environ.get("LANGSMITH_PROJECT", "default")))
    print("Dual tracing: Langfuse + LangSmith")
else:
    print("Langfuse only (LangSmith env 미설정)")

agent.invoke(
    {"messages": [{"role": "user", "content": "대구 날씨?"}]},
    config={"callbacks": callbacks, "tags": ["dual-tracing"]},
)

### flush

Langfuse 4.x 는 OTel BatchSpanProcessor 가 내부에서 배치로 전송한다. 노트북/스크립트가 종료되기 전에 버퍼가 비워지지 않으면 마지막 trace 가 누락될 수 있으므로, 명시적 flush 를 권장한다.

In [ ]:
client.flush()
print("flush 완료")

## 체크리스트

- [ ] `.env` 에 `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` / `LANGFUSE_HOST` 세 줄
- [ ] `from langfuse.langchain import CallbackHandler` (4.x 기준 import 경로)
- [ ] `invoke(config={"callbacks":[handler]})` 로 호출 레벨 주입
- [ ] trace · score · dataset 을 LangSmith 의 run · feedback · dataset 과 매핑
- [ ] self-host 시 **ClickHouse 가 trace 스토어** — 백업·스케일 기준
- [ ] 커스텀 모델은 `Settings → Model pricing` 에서 단가 등록해야 cost 집계에 들어감
- [ ] 다중 백엔드 운영은 callbacks 리스트에 여러 tracer 를 동시 주입
- [ ] 종료 전 `get_client().flush()` 로 OTel 버퍼 플러시

## 다음

- `02_opentelemetry.ipynb` — 벤더 중립 OTel 로 Datadog · Grafana Tempo · Jaeger 에 라우팅
- `06_langsmith/` — LangSmith 전용 파트 (dataset / evaluator / prompt hub)

## 참고

- Langfuse 공식: https://langfuse.com/docs
- Python SDK v4 (OTel 기반): https://langfuse.com/docs/sdk/python/sdk-v3  (v3→v4 통합 가이드)
- LangChain integration: https://langfuse.com/docs/integrations/langchain/tracing
- Self-host: https://langfuse.com/self-hosting
- Model pricing 관리: https://langfuse.com/docs/model-usage-and-cost